# Assignment 7.1: Data Preprocessing

Student: Md Illeas Hossain  
Course: Method of Artificial Intelligence and Machine Learning  
Assignment: 7.1 Data Preprocessing

## Purpose

This notebook preprocesses the United Nations immigration dataset for Canada. The goal is to transform the original worksheet into a clean DataFrame with country as the index, meaningful geographic columns, yearly immigration counts from 1980 through 2013, and a `Total` column containing the sum of immigration across all years.



## 1. Import the required libraries

`pandas` is used to load, clean, transform, and summarize the immigration dataset. `Path` is used to verify that the Excel file is available in the current notebook folder.


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("pandas version:", pd.__version__)


## 2. Load the United Nations immigration data

The workbook contains introductory rows above the actual table, so the first 20 rows are skipped. The last two footer rows are also excluded. The worksheet used for this assignment is Canada by Citizenship.


In [ ]:
file_path = Path("Canada.xlsx")

if not file_path.exists():
    raise FileNotFoundError(
        "Canada.xlsx was not found. Download the instructor's Google Sheet as an Excel (.xlsx) file, "
        "rename it Canada.xlsx, and place it in the same folder as Assignment_7_1.ipynb."
    )

df_can = pd.read_excel(
    file_path,
    sheet_name="Canada by Citizenship",
    skiprows=20,
    skipfooter=2,
    engine="openpyxl"
)

print("Original DataFrame shape:", df_can.shape)
df_can.head()


## 3. Inspect the original structure

The original dataset contains descriptive fields such as `Type`, `Coverage`, numeric geographic codes (`AREA`, `REG`, and `DEV`), geographic names, and yearly immigration counts.


In [ ]:
print("Original columns:")
print(df_can.columns.tolist())

print("\nData types:")
print(df_can.dtypes.head(15))


## 4. Remove columns that are not needed

For the required output, the numeric classification/code columns and the `Type` and `Coverage` fields are unnecessary. The readable geographic names are retained.


In [ ]:
columns_to_drop = ["Type", "Coverage", "AREA", "REG", "DEV"]

df_clean = df_can.drop(
    columns=[c for c in columns_to_drop if c in df_can.columns]
).copy()

print("Shape after dropping unnecessary columns:", df_clean.shape)
df_clean.head()


## 5. Rename columns for readability

The original column names are renamed so that the final DataFrame clearly identifies the country, continent, and region.


In [ ]:
df_clean.rename(
    columns={
        "OdName": "Country",
        "AreaName": "Continent",
        "RegName": "Region"
    },
    inplace=True
)

df_clean.head()


## 6. Standardize column names and prepare the year columns

All column labels are converted to strings. This makes it easy to create a list of year columns from 1980 through 2013 and perform the same cleaning operation on all of them.


In [ ]:
df_clean.columns = [str(col).strip() for col in df_clean.columns]

years = [str(year) for year in range(1980, 2014)]

missing_years = [year for year in years if year not in df_clean.columns]
if missing_years:
    raise ValueError(f"Expected year columns are missing: {missing_years}")

df_clean[years] = (
    df_clean[years]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

print("Year range:", years[0], "to", years[-1])


## 7. Set country as the DataFrame index

The assignment's target layout shows each country as the row label. Therefore, `Country` is moved to the index.


In [ ]:
df_clean["Country"] = df_clean["Country"].astype(str).str.strip()
df_clean.set_index("Country", inplace=True)
df_clean.index.name = "Country"

df_clean.head()


## 8. Calculate total immigration

A new `Total` column is created by summing immigration values across all years from 1980 through 2013 for each country.


In [ ]:
df_clean["Total"] = df_clean[years].sum(axis=1)

df_clean[["Continent", "Region", "DevName", "Total"]].head()


## 9. Reorder the columns to match the required output

The final structure begins with `Continent`, `Region`, and `DevName`, followed by all annual immigration values and the new `Total` column.


In [ ]:
final_columns = ["Continent", "Region", "DevName"] + years + ["Total"]
df_final = df_clean[final_columns].copy()

print("Final DataFrame shape:", df_final.shape)
df_final.head()


## 10. Verify the transformation

The first five countries shown in the assignment example are displayed below. Their totals provide a useful check that the preprocessing steps were performed correctly.


In [ ]:
countries_to_check = [
    "Afghanistan",
    "Albania",
    "Algeria",
    "American Samoa",
    "Andorra"
]

available = [country for country in countries_to_check if country in df_final.index]

verification = df_final.loc[
    available,
    ["Continent", "Region", "DevName", "1980", "1981", "1982", "2011", "2012", "2013", "Total"]
]

verification


For the dataset shown in the assignment instructions, the expected totals for the first five countries are:

- Afghanistan: **58,639**
- Albania: **15,699**
- Algeria: **69,439**
- American Samoa: **6**
- Andorra: **15**

The following cell checks these values when the same 2015 Revision dataset is used.


In [ ]:
expected_totals = {
    "Afghanistan": 58639,
    "Albania": 15699,
    "Algeria": 69439,
    "American Samoa": 6,
    "Andorra": 15
}

for country, expected in expected_totals.items():
    if country in df_final.index:
        actual = int(df_final.loc[country, "Total"])
        print(f"{country:16s} actual={actual:6d} expected={expected:6d} match={actual == expected}")


## 11. Check for missing values and duplicates

These checks help confirm that the final preprocessing result is suitable for further analysis.


In [ ]:
print("Total missing values:", int(df_final.isna().sum().sum()))
print("Duplicate country index values:", int(df_final.index.duplicated().sum()))
print("Number of countries/areas:", len(df_final))


## 12. Save the cleaned dataset

The transformed DataFrame is exported to CSV so that it can be reused in later analysis without repeating the entire preprocessing workflow.


In [ ]:
output_file = Path("canada_immigration_preprocessed.csv")
df_final.to_csv(output_file)

print(f"Saved cleaned dataset to: {output_file.resolve()}")


## Conclusion

The United Nations immigration worksheet was converted from its original reporting format into an analysis-ready DataFrame. Unnecessary classification codes were removed, descriptive columns were renamed, country names were used as the index, annual immigration values were standardized, and a total immigration measure was calculated for 1980–2013. The resulting structure matches the format requested in the assignment and can now be used for visualization, clustering, or other machine-learning tasks.

## References

So, A., So, W., & Nagy, Z. (2020). The applied artificial intelligence workshop: Start working with AI today, to build games, design decision trees, and train your own machine learning models. Packt Publishing.

United Nations, Department of Economic and Social Affairs, Population Division. (2015). International migration flows to and from selected countries: The 2015 revision [Data set]. United Nations.
